# F2 & Supp Fig: Reward-magnitude analyses

Fig. 2b, 2c and Supplementary Fig. 4. Sample efficiency of the large-network PC operating point
across reward magnitudes (Fig. 2c), example learning curves, and the rate and asymptote
sensitivity of every rule, from the reward-magnitude sweep in `data/magrate/` (Supp. Fig. 4) and
from runs at each rule's operating point (Fig. 2b).

In [ ]:
import os
from ssb.utils import batch_inits
from ssb.envs import bandits
from ssb.scripts import run
from ssb.models import ssb, rules
from ssb.analysis.best_params import best_params
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from adjustText import adjust_text

mpl.rcParams['font.family'] = 'Arial'
RULES = ['BCM','OJA','LC','PC','COV','HZ','HR']
magrate_folder = '../../data/magrate/'
C_ACT = "#2171b5"
CACHE_DIR = 'cache'  # simulate_rate_asym / run_efficiency_simulations cache their (expensive) results here

# Sweep grids (see README)
etas   = np.logspace(-6, -1, 6)
alphas = np.logspace(-5, 0, 15)
betas  = np.logspace(-5, 0, 15)
taus   = np.logspace(-3, -1, 4)


def style_ax(ax, fontsize=8):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for spine in ax.spines.values():
        spine.set_linewidth(0.5)
    ax.tick_params(axis="both", labelsize=fontsize, length=2, pad=1.5)
    ax.yaxis.set_tick_params(width=0.5)
    ax.xaxis.set_tick_params(width=0.5)


def rolling_mean(x, w):
    n = (len(x) // w) * w
    return x[:n].reshape(-1, w).mean(axis=1)


def plot_reward_curve_multiple(ax, actions, window, line_alpha=1.0, fill_alpha=0.3,
                                color="blue", label=None, error=False):
    """Plot median (± IQR if error=True) onto an existing axis. Caller controls ylim/styling."""
    smoothed = np.stack([rolling_mean(actions[:, i], window)
                         for i in range(actions.shape[1])], axis=1)
    median = np.median(smoothed, axis=1)
    p25    = np.percentile(smoothed, 25, axis=1)
    p75    = np.percentile(smoothed, 75, axis=1)
    t      = np.arange(smoothed.shape[0]) * window
    ax.plot(t, median, alpha=line_alpha, color=color, lw=0.8, label=label)
    if error:
        ax.fill_between(t, p25, p75, alpha=fill_alpha, color=color, linewidth=0)


# F2: Sample efficiency across reward magnitude (PC)

In [ ]:
# --- 1. DATA GENERATION FUNCTION ---
def run_efficiency_simulations(rewards, rule, size='big', n_samples=20, n_seeds=5, use_cache=True):
    """
    Runs n_samples independent batches of n_seeds seeds (Fig. 2c: 20 x 20), using the
    operating point (eta, alpha, beta, tau) for `rule`/`size` (ssb.analysis.best_params). Cached to cache/sample_efficiency_{rule}_{size}.npz --
    a parameter mismatch (rewards/n_samples/n_seeds) triggers a fresh run.
    """
    rewards = np.asarray(rewards, dtype=float)
    cache_path = os.path.join(CACHE_DIR, f'sample_efficiency_{rule}_{size}.npz')

    if use_cache and os.path.exists(cache_path):
        cached = np.load(cache_path)
        same_params = (np.allclose(cached['rewards'], rewards)
                      and int(cached['n_samples']) == n_samples
                      and int(cached['n_seeds']) == n_seeds)
        if same_params:
            print(f"{rule} ({size}): loaded sample-efficiency data from cache ({cache_path})")
            actions_all = cached['actions']  # (n_rewards, n_samples, timesteps, n_seeds)
            return {reward: [actions_all[i, k] for k in range(n_samples)]
                   for i, reward in enumerate(rewards)}
        print(f"{rule} ({size}): sample-efficiency cache params differ, recomputing")

    eta, alpha, beta, tau = best_params(rule, size)
    print(f"{rule} ({size}): eta={eta:.4g}, alpha={alpha:.4g}, beta={beta:.4g}, tau={tau:.4g}")

    raw_data = {}
    actions_all = None
    for i, reward in enumerate(rewards):
        raw_data[reward] = []
        for k in range(n_samples):
            actions, _, _ = run.run_quick(rule, alpha=alpha, beta=beta, eta_a=eta,
                                          env=bandits.bandit_det, timesteps=20_000, 
                                          size=size, n_seeds=n_seeds, reward=reward,
                                          seed=235+23229*k, tau=tau)
            raw_data[reward].append(actions)
            if actions_all is None:
                actions_all = np.zeros((len(rewards), n_samples) + actions.shape, dtype=np.float32)
            actions_all[i, k] = actions

        print(f"Finished simulating data for reward: {reward}")

    os.makedirs(CACHE_DIR, exist_ok=True)
    np.savez_compressed(cache_path, rewards=rewards, n_samples=n_samples, n_seeds=n_seeds,
                        actions=actions_all)
    return raw_data

def get_trials_to_criterion(actions, window, threshold=0.9, max_trials=20_000):
    """First trial at which the median-across-seeds curve reaches `threshold`."""
    # 1. Bin each seed, as in plot_reward_curve_multiple
    smoothed = np.stack([rolling_mean(actions[:, i], window)
                         for i in range(actions.shape[1])], axis=1)
    
    # 2. Calculate the median across the seeds
    median = np.median(smoothed, axis=1)
    
    # 3. Find when it first crosses the threshold
    above_threshold = median >= threshold
    
    if np.any(above_threshold):
        # multiply by window to convert bin index → actual trial count
        return int(np.argmax(above_threshold)) * window
    else:
        return max_trials

def process_saved_data(raw_data, window=100, threshold=0.9):
    """Trials to criterion for every batch and reward level."""
    processed_results = {}
    for reward, samples_list in raw_data.items():
        criteria_samples = []
        for actions in samples_list:
            criterion = get_trials_to_criterion(actions, window=window, threshold=threshold)
            criteria_samples.append(criterion)
        processed_results[reward] = np.array(criteria_samples)
    return processed_results

def plot_sample_efficiency(results, save_path=None, ylim=None):
    fig, ax = plt.subplots(figsize=(2.0, 1.4))
    for reward, criteria_samples in results.items():
        x_vals = np.full(len(criteria_samples), reward)
        jitter = np.random.normal(0, 0.015 * reward, size=len(criteria_samples))
        ax.scatter(x_vals + jitter, criteria_samples,
                   facecolors='none', edgecolors='k', s=8, alpha=0.7, zorder=2, lw=0.5)
        mean_crit = np.mean(criteria_samples)
        sem_crit  = np.std(criteria_samples) / np.sqrt(len(criteria_samples))
        ax.errorbar(reward, mean_crit, yerr=sem_crit,
                    fmt='_', color='red', markersize=6, capsize=2,
                    markeredgewidth=1, elinewidth=1, zorder=3)
    ax.set_xlabel('Reward size', fontsize=8)
    ax.set_ylabel('Reinforcements\nto criterion', fontsize=8)
    style_ax(ax)
    fig.tight_layout(pad=0.3)
    if ylim:
        ax.set_ylim(ylim)
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Fig. 2c: 20 batches of 20 seeds at r = 0.1-1.0 (cached after the first run)
rewards_to_test = np.linspace(.1, 1., 10)
raw_simulation_data = run_efficiency_simulations(rewards_to_test, 'PC', size='big', n_samples=20, n_seeds=20)

In [ ]:
# Trials to criterion: first 100-trial bin at which the median curve reaches 0.9
current_results = process_saved_data(raw_simulation_data, window=100, threshold=0.9)
plot_sample_efficiency(current_results, save_path='f2_sample_efficiency.svg')

In [ ]:
eta, alpha, beta, tau = best_params("PC", "big")
print(f"PC (big): eta={eta:.4g}, alpha={alpha:.4g}, beta={beta:.4g}, tau={tau:.4g}")

fig, ax = plt.subplots(figsize=(2.2, 1.0))
rewards = np.array([.2, .5, 1.])
for i, reward in enumerate(rewards):
    actions, _, _ = run.run_quick("PC", alpha=alpha, beta=beta, eta_a=eta,
                                  env=bandits.bandit_det, timesteps=20_000,
                                  size="big", n_seeds=300, reward=reward, tau=tau)
    alpha_val = 0.2 + 0.8 * (i / (len(rewards) - 1))
    plot_reward_curve_multiple(ax, actions, window=100,
                               line_alpha=alpha_val, color=C_ACT,
                               error=False, label=f"{reward:.1f}")

ax.set_xlabel("Timestep", fontsize=8)
ax.set_ylabel("P(optimal)", fontsize=8)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=7, frameon=False, loc="lower right", title="Reward", title_fontsize=7)
style_ax(ax)
fig.tight_layout(pad=0.3)
plt.savefig("f2_pc_magreward_big.svg", dpi=300, bbox_inches="tight")
plt.show()

# Supp Fig: reward-magnitude rate & asymptote sensitivity

Full-coverage, relative-criterion (P(optimal) >= crit_frac * asymptote, crit_frac=0.9) rate and
asymptote sensitivity, computed directly from the magnitude-sweep aggregates in
`data/magrate/mag_{rule}_{arch}.npz` (one file per rule x arch). Since crit_frac <= 1, every cell
has a genuine `t_crit` crossing (a mean's own tail always contains a value >= the mean) -- there
is no `censored` field, and `t_crit` is never a lower bound.

    rate[i,j]  = 1 / t_crit[i,j]
    fit  rate ~ d_rate_dR * R + b     (needs >= 3 usable reward levels)

    asym[i,j]  = asymptote[i,j]       (always defined)
    fit  asym ~ d_asym_dR * R + b     (needs >= 3 usable reward levels)

In [ ]:
MAG_RULES = ['BCM','OJA','LC','PC','COV','HZ','HR']
ARCHS = ['small', 'big']

In [ ]:
N_REWARDS = 5
REWARD_CMAP = plt.cm.Blues


def reward_color(j, n=N_REWARDS):
    return REWARD_CMAP(0.35 + 0.55 * j / (n - 1))


C_SMALL = '#2a78d6'
C_BIG = '#eb6834'
ARCH_COLOR = {'small': C_SMALL, 'big': C_BIG}

## Fit helper

In [ ]:
def fit_slope(x, y, min_n=3):
    """OLS slope/intercept/r2 of y on x, ignoring non-finite pairs."""
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    if ok.sum() < min_n:
        return np.nan, np.nan, np.nan, int(ok.sum())
    xs, ys = x[ok], y[ok]
    coef = np.polyfit(xs, ys, 1)
    pred = np.polyval(coef, xs)
    ss_res = float(np.sum((ys - pred) ** 2))
    ss_tot = float(np.sum((ys - ys.mean()) ** 2))
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    return float(coef[0]), float(coef[1]), r2, int(ok.sum())

## Load every rule/arch, compute full-coverage sensitivities

In [ ]:
records = []  # one dict per cell with >=3 usable reward levels on both fits

for rule in MAG_RULES:
    for arch in ARCHS:
        d = np.load(f'{magrate_folder}mag_{rule}_{arch}.npz', allow_pickle=True)
        rewards = np.asarray(d['rewards'], dtype=float)
        t_crit = d['t_crit'].astype(float)
        asymptote = d['asymptote'].astype(float)
        cell_idx = d['cell_idx']

        for i in range(t_crit.shape[0]):
            with np.errstate(divide='ignore'):
                rate = np.where(t_crit[i] > 0, 1.0 / np.maximum(t_crit[i], 1e-9), np.nan)
            d_rate_dR, _, r2_rate, n_rate = fit_slope(rewards, rate)
            d_asym_dR, _, r2_asym, n_asym = fit_slope(rewards, asymptote[i])
            if n_rate < 3 or n_asym < 3:
                continue
            records.append(dict(rule=rule, arch=arch, idx=i, cell_idx=cell_idx[i],
                                asymptote=asymptote[i], t_crit=t_crit[i],
                                d_rate_dR=d_rate_dR, r2_rate=r2_rate, n_rate=n_rate,
                                d_asym_dR=d_asym_dR, r2_asym=r2_asym, n_asym=n_asym))

print(f'{len(records)} cells with usable fits (full coverage, relative criterion)')

## Three illustrative example cells

Searched directly from `records`, restricted to cells with all 5 reward levels usable
(`n_rate == n_asym == 5`, i.e. rate is finite at every level) so the curves plot cleanly:

- **Example** -- strong on both axes: `r2_rate`, `r2_asym` > 0.9, `d_rate_dR` and `d_asym_dR` both
  positive, ranked by their product.
- **Reward-insensitive** -- both slopes near zero and the asymptote isn't just stuck at zero
  (mean asymptote > 0.15), ranked by how close both slopes are to 0.
- **Rate-sensitive, asymptote-flat** -- strong `r2_rate` with `d_asym_dR` pinned near zero,
  ranked by `|d_rate_dR|`.

In [ ]:
def clean(r):
    return r['n_rate'] == 5 and r['n_asym'] == 5


strong = [r for r in records if clean(r) and r['r2_rate'] > 0.9 and r['r2_asym'] > 0.9
         and r['d_rate_dR'] > 0 and r['d_asym_dR'] > 0]
strong.sort(key=lambda r: -(r['d_rate_dR'] * r['d_asym_dR']))

flat = [r for r in records if clean(r) and np.isfinite(r['r2_rate']) and np.isfinite(r['r2_asym'])
       and abs(r['d_rate_dR']) < 0.0004 and abs(r['d_asym_dR']) < 0.04
       and r['asymptote'].mean() > 0.15]
flat.sort(key=lambda r: abs(r['d_rate_dR']) + abs(r['d_asym_dR']))

rate_only = [r for r in records if clean(r) and r['r2_rate'] > 0.85 and abs(r['d_asym_dR']) < 0.03]
rate_only.sort(key=lambda r: -abs(r['d_rate_dR']))

EXAMPLES = {
    'Example': strong[0],
    'Reward-insensitive': flat[0],
    'Rate-sensitive, asymptote-flat': rate_only[0],
}
for label, r in EXAMPLES.items():
    print(f"{label:34s} {r['rule']}/{r['arch']}  cell {r['idx']:4d}  "
         f"d_rate_dR={r['d_rate_dR']:.5f}  d_asym_dR={r['d_asym_dR']:.4f}")

## Load the binned curves for those three cells, and their per-reward rate

In [ ]:
def load_curve_data(rule, arch, i):
    d = np.load(f'{magrate_folder}mag_{rule}_{arch}.npz', allow_pickle=True)
    rewards = np.asarray(d['rewards'], dtype=float)
    curves = d['curves'][i]                     # (rewards, seeds, bins)
    med = np.median(curves, axis=1)
    p25 = np.percentile(curves, 25, axis=1)
    p75 = np.percentile(curves, 75, axis=1)
    bin_width = int(d['bin_width'])
    t_axis = (np.arange(med.shape[-1]) + 1) * bin_width
    return dict(rewards=rewards, med=med, p25=p25, p75=p75, t_axis=t_axis,
               bin_width=bin_width, crit_frac=float(d['crit_frac']))


def rate_array(r):
    with np.errstate(divide='ignore'):
        return np.where(r['t_crit'] > 0, 1.0 / np.maximum(r['t_crit'], 1e-9), np.nan)


def hp_str(cell_idx):
    """Format a (eta_idx, tau_idx, alpha_idx, beta_idx) cell_idx tuple as hyperparameter values."""
    eta_idx, tau_idx, alpha_idx, beta_idx = cell_idx
    eta, alpha, beta = etas[eta_idx], alphas[alpha_idx], betas[beta_idx]
    tau = taus[tau_idx] if tau_idx >= 0 else 0.
    return f"eta={eta:.3g}, alpha={alpha:.3g}, beta={beta:.3g}, tau={tau:.3g}"


for r in EXAMPLES.values():
    r['curve_data'] = load_curve_data(r['rule'], r['arch'], r['idx'])
    r['rate'] = rate_array(r)

## Shared y-domains for the rate/asymptote mini-panels

So the three examples' fit panels sit on the same scale and heights are directly comparable,
instead of each auto-zooming to its own data range.

In [ ]:
all_rates = np.concatenate([r['rate'][np.isfinite(r['rate'])] for r in EXAMPLES.values()])
all_asyms = np.concatenate([r['asymptote'] for r in EXAMPLES.values()])

# Rate spans multiple orders of magnitude across cells (t_crit ranges from ~100
# to ~100,000 steps), so a linear axis makes real differences look flat --
# pad multiplicatively (log space) instead of additively, and plot log-scale.
log_rates = np.log10(all_rates[all_rates > 0])
rate_pad = (log_rates.max() - log_rates.min()) * 0.15 or 0.3
RATE_DOMAIN = (10 ** (log_rates.min() - rate_pad), 10 ** (log_rates.max() + rate_pad))

asym_pad = (all_asyms.max() - all_asyms.min()) * 0.15 or 0.1
ASYM_DOMAIN = (max(0.0, all_asyms.min() - asym_pad), min(1.05, all_asyms.max() + asym_pad))

print('RATE_DOMAIN ', RATE_DOMAIN)
print('ASYM_DOMAIN ', ASYM_DOMAIN)

## Figure: per-cell walkthrough (curves -> extraction -> fits)

For each example: binned P(optimal) (median +/- IQR across 20 seeds) per reward level, with the
0.9-of-asymptote criterion crossing marked (dot; the criterion is relative to each level's own
asymptote) and the final-tenth-of-bins asymptote window shaded
(tick = mean); then rate (1/t_crit) and asymptote each fit linearly against reward.

In [ ]:
from matplotlib.ticker import FuncFormatter, MaxNLocator


def curve_crossing(t_arr, y_arr, thresh):
    """First point (interpolated) where `y_arr` reaches `thresh`, so the
    marker sits exactly on the drawn polyline -- not on the raw, unsmoothed
    t_crit, which can fall well off the (smoothed) displayed curve."""
    above = y_arr >= thresh
    if not above.any():
        return None
    k = int(np.argmax(above))
    if k == 0:
        return float(t_arr[0]), float(y_arr[0])
    t0, t1, y0, y1 = t_arr[k - 1], t_arr[k], y_arr[k - 1], y_arr[k]
    frac = 0.0 if y1 == y0 else (thresh - y0) / (y1 - y0)
    return float(t0 + frac * (t1 - t0)), float(thresh)


def plot_example(label, r, save=None, rate_domain=None, asym_domain=None,
                 max_display_steps=50_000, smooth_bins=10):
    rate_domain = rate_domain or RATE_DOMAIN
    asym_domain = asym_domain or ASYM_DOMAIN
    cd = r['curve_data']
    rewards, med, p25, p75 = cd['rewards'], cd['med'], cd['p25'], cd['p75']
    t_axis, bin_width, crit_frac = cd['t_axis'], cd['bin_width'], cd['crit_frac']
    n_bins = med.shape[-1]
    tail_bins = max(1, n_bins // 10)
    tail_start = (n_bins - tail_bins) * bin_width
    t_max = t_axis[-1]
    t_display_max = min(t_max, max_display_steps)

    fig, axes = plt.subplots(1, 3, figsize=(6.0, 1.5),
                             gridspec_kw=dict(width_ratios=[2.1, 1, 1], wspace=0.7))

    # --- panel 1: annotated curves ---
    # Rolling-smoothed (block mean over `smooth_bins` bins) and truncated to
    # max_display_steps -- the raw per-100-step-bin curve is too jittery to
    # read at this scale, and cells only need ~50k of a 100k-step run to show
    # their actual dynamics; the rest is flat tail. `t_s` labels each smoothed
    # block by its *center* (not its right edge) so a block's plotted x
    # position matches when, on average, its value actually occurred --
    # otherwise the criterion-crossing dot (below) can land up to a whole
    # block ahead of where the smoothed curve visually reaches that height.
    # The dot itself is placed by interpolating directly on the displayed
    # (smoothed) curve rather than on the raw, unsmoothed t_crit, so it
    # always sits exactly on the drawn line rather than floating off it.
    ax = axes[0]
    if tail_start < t_display_max:
        ax.axvspan(tail_start, t_display_max, color='black', alpha=0.05, lw=0)
    for j, rw in enumerate(rewards):
        c = reward_color(j)
        level_thresh = crit_frac * r['asymptote'][j]  # relative criterion: own asymptote, not a fixed 0.9
        med_s = rolling_mean(med[j], smooth_bins)
        p25_s = rolling_mean(p25[j], smooth_bins)
        p75_s = rolling_mean(p75[j], smooth_bins)
        t_s = (np.arange(len(med_s)) + 0.5) * bin_width * smooth_bins
        keep = t_s <= t_display_max
        ax.fill_between(t_s[keep], p25_s[keep], p75_s[keep], color=c, alpha=0.12, linewidth=0)
        ax.plot(t_s[keep], med_s[keep], color=c, lw=0.9)
        ax.hlines(level_thresh, 0, t_display_max, color=c, lw=0.4, linestyle='--', alpha=0.4)
        cross = curve_crossing(t_s, med_s, level_thresh)
        if cross is not None and cross[0] <= t_display_max:
            ax.plot(*cross, 'o', color=c, ms=4.5, mec='white', mew=0.6, zorder=4)
        ax.plot([t_display_max * 0.965, t_display_max], [r['asymptote'][j]] * 2, color=c, lw=2.4, solid_capstyle='butt')
    ax.set_xlim(0, t_display_max); ax.set_ylim(-0.03, 1.08)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x/1000:.0f}k' if x else '0'))
    ax.set_xlabel('Timestep', fontsize=8); ax.set_ylabel('P(optimal)', fontsize=8)
    style_ax(ax)

    # --- panel 2: rate vs reward ---
    # log scale: t_crit spans ~100-100,000 steps across cells, so rate (1/t_crit)
    # spans several orders of magnitude -- a linear axis compresses real
    # differences (and real slopes) down to what looks like a flat line near 0.
    ax = axes[1]
    m, b, *_ = fit_slope(rewards, r['rate'])
    xs_fit = np.linspace(0.15, 1.02, 50)
    fit_line = m * xs_fit + b
    ok = fit_line > 0
    ax.plot(xs_fit[ok], fit_line[ok], color='#aaaaaa', lw=0.8)
    ax.scatter(rewards, r['rate'], c=[reward_color(j) for j in range(len(rewards))],
              s=10, edgecolors='white', linewidths=0.3, zorder=3)
    ax.set_yscale('log')
    ax.set_xlim(0.1, 1.05); ax.set_ylim(*rate_domain)
    ax.set_xlabel('Reward', fontsize=8); ax.set_ylabel('Rate (1/t_crit)', fontsize=8)
    ax.set_title(f'd_rate_dR={r["d_rate_dR"]:.1e}', fontsize=8, pad=2)
    style_ax(ax)

    # --- panel 3: asymptote vs reward ---
    ax = axes[2]
    m, b, *_ = fit_slope(rewards, r['asymptote'])
    xs_fit_lin = np.array([0.15, 1.02])
    ax.plot(xs_fit_lin, m * xs_fit_lin + b, color='#aaaaaa', lw=0.8)
    ax.scatter(rewards, r['asymptote'], c=[reward_color(j) for j in range(len(rewards))],
              s=10, edgecolors='white', linewidths=0.3, zorder=3)
    ax.set_xlim(0.1, 1.05); ax.set_ylim(*asym_domain)
    ax.set_xlabel('Reward', fontsize=8); ax.set_ylabel('Asymptote', fontsize=8)
    ax.set_title(f'd_asym_dR={r["d_asym_dR"]:.3f}', fontsize=8, pad=2)
    style_ax(ax)

    if 'cell_idx' in r:
        hp_label = hp_str(r['cell_idx'])
    else:
        hp_label = f"eta={r['eta']:.3g}, alpha={r['alpha']:.3g}, beta={r['beta']:.3g}, tau={r['tau']:.3g}"
    fig.suptitle(f"{label}  ({r['rule']}_{r['arch']}: {hp_label})", fontsize=9, y=1.15)
    fig.tight_layout(pad=0.3)
    if save:
        fig.savefig(save, dpi=300, bbox_inches='tight')
    return fig


SLUG = {'Example': 'example', 'Reward-insensitive': 'flat', 'Rate-sensitive, asymptote-flat': 'rate_only'}
for label, r in EXAMPLES.items():
    plot_example(label, r, save=f'sfig_magrate_{SLUG[label]}.svg')

## Figure: population scatter, all cells faceted by rule

Same x/y as the walkthrough above -- `d_rate_dR` vs `d_asym_dR` -- one point per cell, colored
by architecture. The three examples are ringed and labelled in their rule's panel.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(7.5, 3.2))
axes = axes.ravel()

all_rate_vals = np.array([r['d_rate_dR'] for r in records])
all_asym_vals = np.array([r['d_asym_dR'] for r in records])
xmax = np.abs(all_rate_vals).max() * 1.08
ymax = np.abs(all_asym_vals).max() * 1.08

for k, rule in enumerate(MAG_RULES):
    ax = axes[k]
    rows = [r for r in records if r['rule'] == rule]
    for arch in ARCHS:
        xs = [r['d_rate_dR'] for r in rows if r['arch'] == arch]
        ys = [r['d_asym_dR'] for r in rows if r['arch'] == arch]
        # rasterized: with ~9000 points across these 8 panels, a plain vector
        # scatter makes the saved .svg enormous (one element per point).
        ax.scatter(xs, ys, s=5, color=ARCH_COLOR[arch], alpha=0.35, linewidths=0,
                  label=arch, rasterized=True)
    ax.axhline(0, color='#c3c2b7', lw=0.5); ax.axvline(0, color='#c3c2b7', lw=0.5)
    ax.set_xlim(-xmax, xmax); ax.set_ylim(-ymax, ymax)
    ax.set_title(f'{rule}  (n={len(rows)})', fontsize=9, pad=2)
    style_ax(ax)
    if k % 4 == 0:
        ax.set_ylabel('Asymptote sensitivity', fontsize=8)
    if k >= 4:
        ax.set_xlabel('Rate sensitivity', fontsize=8)

    for label, r in EXAMPLES.items():
        if r['rule'] != rule:
            continue
        ax.scatter([r['d_rate_dR']], [r['d_asym_dR']], s=40, facecolors='none',
                  edgecolors='black', linewidths=1.0, zorder=5)
        ax.annotate(label.split(',')[0], (r['d_rate_dR'], r['d_asym_dR']),
                   fontsize=7, xytext=(3, 3), textcoords='offset points')

axes[0].legend(fontsize=7, frameon=False, loc='upper left', markerscale=2)
axes[-1].axis('off')
fig.tight_layout(pad=0.4)
fig.savefig('sfig_magrate_population.svg', dpi=300, bbox_inches='tight')

## Figure 2: rate vs asymptote sensitivity at each rule's best fittable parameters

One point per rule, small and big architectures plotted as separate figures below. Hyperparameters
are `best_params(rule, arch)`, the operating points selected from the det sweep (Methods,
"Reward-seeking operating point"). The `data/magrate` sweep capped activities at 3, so the runs
here are simulated again without the cap, with `run.run_quick` as elsewhere in this notebook
(5 reward levels, 50 seeds, 100,000 steps; Methods, "Rate and asymptote sensitivity"). Rule labels are placed with `adjustText` (leader line drawn
only when a label has to move off its point).

In [ ]:
def simulate_rate_asym(rule, arch, rewards=(0.2, 0.4, 0.6, 0.8, 1.0), n_seeds=50,
                       timesteps=100_000, bin_width=100, crit_frac=0.9, use_cache=True):
    """Reward-magnitude runs at this rule/arch's operating point, without the activity cap.

    Computes t_crit (first bin at which the median curve reaches crit_frac x its own asymptote),
    the asymptote (mean of the final 10% of bins), rate = 1/t_crit, and the two sensitivities.
    Cached to cache/rate_asym_{rule}_{arch}.npz (100k steps x 50 seeds x 5 reward levels); a
    change in any simulation parameter triggers a fresh run."""
    rewards = np.asarray(rewards, dtype=float)
    cache_path = os.path.join(CACHE_DIR, f'rate_asym_{rule}_{arch}.npz')

    if use_cache and os.path.exists(cache_path):
        cached = np.load(cache_path)
        same_params = (int(cached['n_seeds']) == n_seeds and int(cached['timesteps']) == timesteps
                      and int(cached['bin_width']) == bin_width
                      and float(cached['crit_frac']) == crit_frac
                      and np.allclose(cached['rewards'], rewards)
                      # also require the cached run to use the current best_params cell
                      and np.allclose([float(cached[k]) for k in ('eta', 'alpha', 'beta', 'tau')],
                                      best_params(rule, arch)))
        if same_params:
            print(f"{rule} ({arch}): loaded from cache ({cache_path})")
            curve_data = dict(rewards=cached['rewards'], med=cached['med'], p25=cached['p25'],
                              p75=cached['p75'], t_axis=cached['t_axis'],
                              bin_width=int(cached['bin_width']), crit_frac=float(cached['crit_frac']))
            return dict(rule=rule, arch=arch, eta=float(cached['eta']), alpha=float(cached['alpha']),
                       beta=float(cached['beta']), tau=float(cached['tau']),
                       asymptote=cached['asymptote'], t_crit=cached['t_crit'], rate=cached['rate'],
                       curve_data=curve_data, d_rate_dR=float(cached['d_rate_dR']),
                       d_asym_dR=float(cached['d_asym_dR']), r2_rate=float(cached['r2_rate']),
                       r2_asym=float(cached['r2_asym']), n_rate=int(cached['n_rate']),
                       n_asym=int(cached['n_asym']))
        print(f"{rule} ({arch}): cache params differ, recomputing")

    eta, alpha, beta, tau = best_params(rule, arch)
    print(f"{rule} ({arch}): eta={eta:.4g}, alpha={alpha:.4g}, beta={beta:.4g}, tau={tau:.4g}")

    n_bins = timesteps // bin_width
    curves = np.zeros((len(rewards), n_seeds, n_bins))
    for j, reward in enumerate(rewards):
        actions, _, _ = run.run_quick(rule, alpha=alpha, beta=beta, eta_a=eta,
                                      env=bandits.bandit_det, timesteps=timesteps,
                                      size=arch, n_seeds=n_seeds, reward=reward, tau=tau)
        curves[j] = actions[:n_bins * bin_width].reshape(n_bins, bin_width, n_seeds).mean(axis=1).T
        print(f"    reward={reward:.1f} done")

    med = np.median(curves, axis=1)              # (rewards, bins)
    tail = max(1, n_bins // 10)
    asymptote = med[:, -tail:].mean(axis=-1)      # (rewards,)
    above = med >= crit_frac * asymptote[:, None]
    ever = above.any(axis=-1)
    first_bin = np.argmax(above, axis=-1)
    t_crit = np.where(ever, first_bin * bin_width, timesteps).astype(int)

    with np.errstate(divide='ignore'):
        rate = np.where(t_crit > 0, 1.0 / np.maximum(t_crit, 1e-9), np.nan)
    d_rate_dR, _, r2_rate, n_rate = fit_slope(rewards, rate)
    d_asym_dR, _, r2_asym, n_asym = fit_slope(rewards, asymptote)

    p25 = np.percentile(curves, 25, axis=1)
    p75 = np.percentile(curves, 75, axis=1)
    t_axis = (np.arange(n_bins) + 1) * bin_width

    os.makedirs(CACHE_DIR, exist_ok=True)
    np.savez(cache_path, eta=eta, alpha=alpha, beta=beta, tau=tau,
            rewards=rewards, med=med, p25=p25, p75=p75, t_axis=t_axis,
            bin_width=bin_width, crit_frac=crit_frac, n_seeds=n_seeds, timesteps=timesteps,
            asymptote=asymptote, t_crit=t_crit, rate=rate,
            d_rate_dR=d_rate_dR, d_asym_dR=d_asym_dR, r2_rate=r2_rate, r2_asym=r2_asym,
            n_rate=n_rate, n_asym=n_asym)

    curve_data = dict(rewards=rewards, med=med, p25=p25, p75=p75, t_axis=t_axis,
                      bin_width=bin_width, crit_frac=crit_frac)
    print(f"    -> d_rate_dR={d_rate_dR:.5f}  d_asym_dR={d_asym_dR:.4f}  "
         f"(n_rate={n_rate}, n_asym={n_asym})")
    return dict(rule=rule, arch=arch, eta=eta, alpha=alpha, beta=beta, tau=tau,
               asymptote=asymptote, t_crit=t_crit, rate=rate, curve_data=curve_data,
               d_rate_dR=d_rate_dR, d_asym_dR=d_asym_dR, r2_rate=r2_rate, r2_asym=r2_asym,
               n_rate=n_rate, n_asym=n_asym)


best_cells = []
for rule in RULES:
    for arch in ARCHS:
        best_cells.append(simulate_rate_asym(rule, arch))


def plot_rate_asym_scatter(arch, save, title=None, figsize=(2.0, 1.8), rate_linthresh=1e-4):
    """Rate vs asymptote sensitivity, one point per rule, for a single
    architecture -- non-overlapping direct labels (adjustText repels them and
    draws a leader line back to the point when it has to move). Several rules
    tend to cluster near the origin, so this needs enough figure area and a
    strong enough push (expand/force_*) for adjustText to actually separate
    them rather than leaving overlaps.

    Rate sensitivity (d_rate_dR) spans several orders of magnitude but can be
    positive or negative and near zero, so it's symlog-scaled (plain log
    can't take signed/zero values) -- otherwise the small differences between
    rules collapse into one cluster near 0."""
    fig, ax = plt.subplots(figsize=figsize)
    rows = [r for r in best_cells if r['arch'] == arch
           and np.isfinite(r['d_rate_dR']) and np.isfinite(r['d_asym_dR'])]
    skipped = [r['rule'] for r in best_cells if r['arch'] == arch and r not in rows]
    if skipped:
        print(f"{arch}: skipping {skipped} (no usable rate/asymptote fit)")

    texts = []
    xs = [r['d_rate_dR'] for r in rows]
    ys = [r['d_asym_dR'] for r in rows]
    ax.scatter(xs, ys, s=18, color=ARCH_COLOR[arch], edgecolors='white', linewidths=0.4, zorder=3)
    for r in rows:
        texts.append(ax.text(r['d_rate_dR'], r['d_asym_dR'], r['rule'],
                             fontsize=8, color=ARCH_COLOR[arch]))
    adjust_text(texts, ax=ax,
               arrowprops=dict(arrowstyle='-', color='#999999', lw=0.4, shrinkA=2, shrinkB=2),
               expand=(1.8, 2.2), force_text=(0.4, 0.6), force_points=(0.3, 0.5))
    ax.axhline(0, color='#c3c2b7', lw=0.5); ax.axvline(0, color='#c3c2b7', lw=0.5)
    ax.set_xscale('symlog', linthresh=rate_linthresh)
    ax.set_xlabel('Rate sensitivity', fontsize=8)
    ax.set_ylabel('Asymptote sensitivity', fontsize=8)
    ax.set_title(title or arch, fontsize=9)
    style_ax(ax)
    fig.tight_layout(pad=0.3)
    fig.savefig(save, dpi=300, bbox_inches='tight')


plot_rate_asym_scatter('small', save='sfig_magrate_best_params_small.svg', title='small networks')
plot_rate_asym_scatter('big', save='f2_magrate_best_params_big.svg', title='big networks')